# Publication copy

Notebook outputs were cleared for distribution. Scientific settings are preserved; see `../docs/REPRODUCIBILITY.md` before interpreting numerical intervals. Run with the notebook working directory set to `plot/`.


# PRL Free SF Eddington-ratio constraints

Version 22 simplifies axis titles and further enlarges tick and title fonts. Data and mark styles are unchanged.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, LogFormatterMathtext, NullFormatter, NullLocator, LogLocator

MM_TO_INCH = 1.0 / 25.4
WIDTH_MM = 85.0
HEIGHT_MM = 70.0
MDOT_FACTOR = 9.73
Y_MIN, Y_MAX = 10.0**-12, 10.0**2
OUTPUT_DIR = Path(os.environ.get("PRL_FIGURE_OUTPUT_DIR", ".")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_STEM = OUTPUT_DIR / "prl_free_sf_accretion_rate_v22"

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "custom",
    "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic",
    "mathtext.bf": "Times New Roman:bold",
    "font.size": 7.2,
    "axes.labelsize": 10.5,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "xtick.major.size": 3.0,
    "ytick.major.size": 4.5,
    "ytick.minor.size": 2.4,
    "ytick.minor.width": 0.6,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 6.6,
    "legend.frameon": False,
    "lines.linewidth": 1.44,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.dpi": 600,
})


## Source data

The latest supplied constraints are mapped to Free SF (blue/filled). All six Eddington-ratio rows are included. Fixed SF is removed as requested; mass and abundance remain outside the plot scope. Values are log10 quantities with asymmetric 68% errors or upper limits. The 85 × 70 mm layout, logarithmic transform, and linked right axis are unchanged.

In [ ]:
redshift_bins = np.array(["6–8", "9", "10", "11", "12.5", "13.5–1000"])
series = {
    "Free astrophysics": {
        "log10_center": np.array([np.nan, np.nan, -1.8, -0.9, -1.1, np.nan]),
        "minus_dex": np.array([np.nan, np.nan, 6.5, 2.4, 3.0, np.nan]),
        "plus_dex": np.array([np.nan, np.nan, 1.4, 1.0, 2.2, np.nan]),
        "log10_limit": np.array([-3.8, -4.4, np.nan, np.nan, np.nan, -2.3]),
        "color": "#4C78A8", "markerface": "#4C78A8",
    },
}


## Transform and validate

For a central value c and dex errors d-/d+, bounds are 10**(c-d-) and 10**(c+d+). Upper limits are transformed directly. Arrow lengths are purely graphical and do not encode an uncertainty or lower bound.

In [ ]:
def log_interval_to_linear(log10_center, minus_dex, plus_dex):
    log10_center = np.asarray(log10_center, dtype=float)
    minus_dex = np.asarray(minus_dex, dtype=float)
    plus_dex = np.asarray(plus_dex, dtype=float)
    assert log10_center.shape == minus_dex.shape == plus_dex.shape
    assert np.all(np.isfinite(log10_center))
    assert np.all(np.isfinite(minus_dex)) and np.all(np.isfinite(plus_dex))
    assert np.all(minus_dex >= 0.0) and np.all(plus_dex >= 0.0)

    center = np.power(10.0, log10_center)
    lower = np.power(10.0, log10_center - minus_dex)
    upper = np.power(10.0, log10_center + plus_dex)
    assert np.all(0.0 < lower) and np.all(lower <= center) and np.all(center <= upper)
    yerr = np.vstack((center - lower, upper - center))
    return center, yerr, lower, upper

def lambda_to_mdot(values):
    return MDOT_FACTOR * np.asarray(values)

def mdot_to_lambda(values):
    return np.asarray(values) / MDOT_FACTOR

for values in series.values():
    measured = np.isfinite(values["log10_center"])
    limited = np.isfinite(values["log10_limit"])
    assert np.all(measured ^ limited)
    assert measured.sum() == limited.sum() == 3
    center, yerr, lower, upper = log_interval_to_linear(
        values["log10_center"][measured], values["minus_dex"][measured], values["plus_dex"][measured])
    values.update(measured=measured, limited=limited, center=center, yerr=yerr, lower=lower, upper=upper)
assert list(series) == ["Free astrophysics"]
assert series["Free astrophysics"]["log10_limit"][1] == -4.4
np.testing.assert_allclose(mdot_to_lambda(lambda_to_mdot(np.array([1e-15, 1.0, 100.0]))), [1e-15, 1.0, 100.0], atol=0)
print("Validated 3 central values with 68% uncertainties and 3 upper limits.")


## Draw and export

Times New Roman text and math; labeled y ticks every two decades, inward logarithmic minor ticks; all spines 0.8 pt. Horizontal tick locations and lengths are preserved. Data marks, arrow length, and arrowhead size remain unchanged. No legend or formula annotation.

Tick labels are 8.5 pt; the horizontal axis title is 10.5 pt and vertical axis titles are 9.5 pt. Axis titles use a space instead of a comma.

In [ ]:
x = np.arange(len(redshift_bins), dtype=float)
offsets = {"Free astrophysics": -0.065}

fig, ax = plt.subplots(figsize=(WIDTH_MM * MM_TO_INCH, HEIGHT_MM * MM_TO_INCH))
fig.subplots_adjust(left=0.16, right=0.85, bottom=0.17, top=0.95)

for label, values in series.items():
    ax.errorbar(
        x[values["measured"]] + offsets[label],
        values["center"],
        yerr=values["yerr"],
        fmt="o",
        markersize=3.6,
        markerfacecolor=values["markerface"],
        markeredgecolor=values["color"],
        markeredgewidth=1.17,
        ecolor=values["color"],
        elinewidth=1.35,
        capsize=2.1,
        capthick=1.35,
        alpha=0.98,
        clip_on=True,
        zorder=2,
        label=label,
    )

for label, values in series.items():
    for xi, limit in zip(x[values["limited"]] + offsets[label], values["log10_limit"][values["limited"]]):
        y = 10.0**limit
        ax.plot([xi], [y], marker="_", markersize=4.2, markeredgewidth=1.35,
                color=values["color"], alpha=0.98)
        ax.annotate("", xy=(xi, 10.0**(limit - 2.5)), xytext=(xi, y),
                    arrowprops=dict(arrowstyle="->", color=values["color"],
                                    lw=1.35, mutation_scale=9, shrinkA=0, shrinkB=0))

if any(np.any(values["center"] <= 0.0) for values in series.values()):
    raise ValueError("A logarithmic y-axis requires strictly positive Eddington ratios.")
ax.set_yscale("log")
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_xlim(-0.45, len(redshift_bins) - 0.55)
ax.set_xticks(x, redshift_bins)
ax.set_xlabel("$z$")
ax.set_ylabel(r"Eddington ratio $\lambda_{\mathrm{E}}$", fontsize=9.5, labelpad=2)

left_ticks = np.power(10.0, np.array([-12, -10, -8, -6, -4, -2, 0, 2], dtype=float))
ax.yaxis.set_major_locator(FixedLocator(left_ticks))
ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(1, 10), numticks=200))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.grid(axis="y", which="major", color="#D8D8D8", linewidth=0.9, alpha=0.75)
ax.set_axisbelow(True)
ax.tick_params(axis="both", which="major", direction="in")
ax.tick_params(axis="y", which="minor", direction="in")
ax.spines["top"].set_visible(True)
ax.tick_params(axis="x", which="major", top=True, labeltop=False, direction="in")

right_ax = ax.secondary_yaxis("right", functions=(lambda_to_mdot, mdot_to_lambda))
right_ax.set_ylabel(r"Accretion rate $\dot{m}$", fontsize=9.5, labelpad=2)
right_ticks = np.power(10.0, np.array([-10, -8, -6, -4, -2, 0, 2], dtype=float))
right_ax.yaxis.set_major_locator(FixedLocator(right_ticks))
right_ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
right_ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(1, 10), numticks=200))
right_ax.yaxis.set_minor_formatter(NullFormatter())
right_ax.tick_params(axis="y", which="both", direction="in")
right_ax.spines["right"].set_visible(True)
right_ax.spines["right"].set_linewidth(0.8)

assert len(ax.texts) == 3  # One arrow per 68% upper limit.
assert not any(line.get_linestyle() == "--" for line in ax.lines)
fig.canvas.draw()
np.testing.assert_allclose(np.asarray(right_ax.get_ylim()), lambda_to_mdot(np.asarray(ax.get_ylim())))

pdf_path = OUTPUT_STEM.with_suffix(".pdf")
svg_path = OUTPUT_STEM.with_suffix(".svg")
preview_path = OUTPUT_STEM.with_suffix(".png")
fig.savefig(pdf_path, facecolor="white", metadata={"Title": "Free SF Eddington ratio and accretion rate"})
fig.savefig(svg_path, facecolor="white")
fig.savefig(preview_path, dpi=600, facecolor="white")
plt.show()

clipped_lower_intervals = sum(int(np.count_nonzero(values["lower"] < Y_MIN)) for values in series.values())
{
    "central_values": 3,
    "upper_limits": 3,
    "right_axis_relation": "mdot = 9.73 lambda_E",
    "lower_intervals_below_visible_floor": clipped_lower_intervals,
    "pdf": str(pdf_path),
    "svg": str(svg_path),
}


## Reuse notes

Only Free SF is plotted: three central values with asymmetric 68% errors and three upper limits. The right axis retains mdot = 9.73 lambda_E. Running the notebook exports PDF, SVG and PNG. No intervals extend below the visible floor. Times New Roman is preserved; enlarged tick and title fonts improve readability. The prior Free SF horizontal offset is retained to preserve positioning.